# Modeling

In [ ]:
import io, nltk, os, re, sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter
from wordcloud import WordCloud
from nltk.corpus import stopwords
from symspellpy import SymSpell, Verbosity
from langdetect import detect, LangDetectException
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, classification_report, accuracy_score

In [ ]:
# 1. Persiapan Stemmer (Sastrawi)
nltk.download('stopwords', quiet=True)
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# 2. Persiapan Stopwords (NLTK)
list_stopwords = stopwords.words('indonesian')

kata_penting = {'tidak', 'kurang', 'jangan', 'bukan', 'tapi', 'sangat', 'jauh'}
list_stopwords = [stopword for stopword in list_stopwords if stopword not in kata_penting]

# Stopword kustom (noise words)
list_stopwords.extend([
    'aja', 'amp', 'biar', 'bikin', 'bilang', 'broo', 'coyy', 'cuy', 'd',
    'deh', 'dg', 'dgn', 'dri', 'eh', 'euy', 'gaes', 'ges', 'guys', 'hehe',
    'hehehe', 'hai', 'halo', 'jgn', 'karna', 'ke', 'kok', 'mah', 'n', 'nah',
    'nih', 'nya', 'pas', 'sdh', 'sih', 't', 'tau', 'tuh', 'utk', 'wkwk',
    'wkwkwk', 'ya', 'yah', 'yee', 'yuhuuuu'
])

set_stopwords = set(list_stopwords)
print(
    f"Stopwords siap digunakan.\n"
    f"- Total stopwords: {len(set_stopwords)} kata\n"
    f"- Kata penting yang tidak dihapus (diamankan): {', '.join(sorted(kata_penting))}\n"
    f"- Contoh stopwords: {', '.join(list(set_stopwords)[:10])}"
)

# 3. Kamus Slang (untuk Normalisasi)
from arc.kamus_slang import kamus_slang

print("\nKamus Slang siap digunakan.")
examples = ", ".join([f"{k}->{v}" for k, v in list(kamus_slang.items())[:10]])
print(f"""Total entri slang/normalisasi : {len(kamus_slang)} kata
Contoh entri                  : {examples}
""")

In [ ]:
file_path = "data/csv/all-task-3-detailed-reviews.csv"

if not os.path.exists(file_path):
    print(f"File tidak ditemukan: {file_path}")
    exit()

try:
    print(f"Membaca file: '{file_path}'")
    df = pd.read_csv(file_path)
    print(f"Total baris data mentah: {len(df)}")

    # 1.1 Penanganan Missing Value (review_text kosong)
    df.dropna(subset=['review_text'], inplace=True)
    df_processed = df.copy()
    print(f"Total baris setelah hapus NaN: {len(df_processed)}")

    # 1.2 PERBAIKAN: Filter Bahasa (Hapus ulasan non-Indonesia)
    def detect_lang(text):
        try:
            return detect(text)
        except LangDetectException:
            return 'error'  # Teks terlalu pendek/aneh

    print("Mendeteksi bahasa ulasan...")
    df_processed['bahasa'] = df_processed['review_text'].apply(detect_lang)

    total_sebelum = len(df_processed)
    df_processed = df_processed[df_processed['bahasa'] == 'id'].copy()
    print(f"Memfilter ulasan non-Indonesia. Dari {total_sebelum} ulasan, {len(df_processed)} ulasan (Bahasa Indonesia) akan diproses.")
    print("--- GATHERING & FILTER AWAL SELESAI ---")

except Exception as e:
    print(f"Error saat membaca/memproses file: {e}")
    print("PROSES BERHENTI. Periksa kembali file CSV dan dependensi.")

In [ ]:
#  4.1 Persiapan Koreksi Typo (SymSpellPy)
print("Menyiapkan Koreksi Typo (SymSpellPy)")
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
corpus_path = 'kamus_corpus_pantai.txt'

# Buat kamus (corpus) dari teks mentah (lebih kaya kata)
all_words_counter = Counter(re.findall(r'\b[a-z]{2,}\b', ' '.join(df_processed['review_text'].str.lower())))
with open(corpus_path, 'w', encoding='utf-8') as f:
    for word, count in all_words_counter.items():
        f.write(f"{word} {count}\n")

sym_spell.load_dictionary(corpus_path, term_index=0, count_index=1)
print(f"SymSpellPy siap (Kamus dari {len(all_words_counter)} kata unik dimuat).")

#  4.2 Mendefinisikan Fungsi Pipeline
# Fungsi 1: Cleaning Dasar (Hapus simbol, angka, dll)
def basic_clean(text):
    text = text.lower() # Case Folding
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Hapus URL
    text = re.sub(r'<.*?>+', '', text) # Hapus HTML
    text = re.sub(r'[^a-z\s]', ' ', text) # Hapus non-huruf (simbol, angka)
    text = re.sub(r'\s+', ' ', text).strip() # Hapus spasi berlebih
    return text

# Fungsi 2: Koreksi Typo (SymSpellPy)
def correct_typos(text):
    tokens = text.split()
    corrected_tokens = []
    for token in tokens:
        suggestions = sym_spell.lookup(token, Verbosity.TOP, max_edit_distance=2)
        if suggestions:
            corrected_tokens.append(suggestions[0].term)
        else:
            corrected_tokens.append(token)
    return ' '.join(corrected_tokens)

# Fungsi 3: Normalisasi Slang (Kamus Manual)
def normalize_slang(text):
    tokens = text.split()
    normalized_tokens = [kamus_slang.get(token, token) for token in tokens]
    return ' '.join(normalized_tokens)

# Fungsi 4: Stemming (Sastrawi)
def stem_text(text):
    return stemmer.stem(text)

# Fungsi 5: Stopword Removal (TERAKHIR)
def remove_stopwords(text):
    tokens = text.split()
    stopped_tokens = [word for word in tokens if word not in set_stopwords]
    return ' '.join(stopped_tokens)

print("Semua Fungsi Pipeline Siap Digunakan ")

In [ ]:
print("Memulai proses preprocessing (Clean, Typo, Slang, Stem, Stopword)...")

def count_words(text):
    return len(text.split())

# Langkah 1/5: Cleaning Dasar
print("\nLangkah 1/5: Cleaning Dasar...")
df_processed['clean_1_basic'] = df_processed['review_text'].apply(basic_clean)
before = df_processed['review_text'].apply(count_words).sum()
after = df_processed['clean_1_basic'].apply(count_words).sum()
print(f"Jumlah kata sebelum: {before:,}, setelah: {after:,}. Perubahan: {after - before:+,} kata.")

# Langkah 2/5: Koreksi Typo
print("\nLangkah 2/5: Koreksi Typo...")
df_processed['clean_2_typo'] = df_processed['clean_1_basic'].apply(correct_typos)
after_typo = df_processed['clean_2_typo'].apply(count_words).sum()
print(f"Jumlah kata setelah koreksi typo: {after_typo:,}.")

# Langkah 3/5: Normalisasi Slang
print("\nLangkah 3/5: Normalisasi Slang...")
df_processed['clean_3_slang'] = df_processed['clean_2_typo'].apply(normalize_slang)
after_slang = df_processed['clean_3_slang'].apply(count_words).sum()
print(f"Jumlah kata setelah normalisasi slang: {after_slang:,}.")

# Langkah 4/5: Stemming
print("\nLangkah 4/5: Stemming...")
df_processed['clean_4_stem'] = df_processed['clean_3_slang'].apply(stem_text)
after_stem = df_processed['clean_4_stem'].apply(count_words).sum()
print(f"Jumlah kata setelah stemming: {after_stem:,}.")

# Langkah 5/5: Stopword Removal
print("\nLangkah 5/5: Stopword Removal...")
df_processed['final_cleaned_text'] = df_processed['clean_4_stem'].apply(remove_stopwords)
final_count = df_processed['final_cleaned_text'].apply(count_words).sum()
print(f"Jumlah kata setelah stopword removal: {final_count:,}.")

In [ ]:
print("Contoh hasil preprocessing final:")
print(df_processed[['review_text', 'final_cleaned_text']].head())

In [ ]:
print("EXPLORATORY DATA ANALYSIS (EDA)")
sns.set(style="whitegrid")

# 1. Visualisasi Distribusi Rating (Insight data mentah)
rating_counts = df_processed['rating'].value_counts().sort_index()
plt.figure(figsize=(7, 7))
plt.pie(
    rating_counts,
    labels=rating_counts.index.map(lambda r: f'{r} Bintang'),
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('viridis_r', n_colors=5)
)
plt.title('Proporsi Distribusi Rating Ulasan (Data Mentah)')
plt.show()

In [ ]:
# 2. Visualisasi Word Cloud (dari teks paling bersih)
all_text = ' '.join(df_processed['final_cleaned_text'])

if all_text:
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_text)
    plt.figure(figsize=(10, 7))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud (dari Teks Final yang Sudah Bersih)')
    plt.show()
else:
    print("Tidak ada teks untuk dibuat Word Cloud.")

In [ ]:
# 3. Visualisasi Bar Chart Top N-Grams (Bigrams)
print("Membuat Bar Chart Top 20 Bigrams (Frasa 2-Kata)...")
vec_bigrams = CountVectorizer(ngram_range=(2, 2), min_df=5)
bigram_matrix = vec_bigrams.fit_transform(df_processed['final_cleaned_text'])
bigram_counts = pd.DataFrame(
    bigram_matrix.toarray(),
    columns=vec_bigrams.get_feature_names_out()
).sum().sort_values(ascending=False)
top_20_bigrams = bigram_counts.head(20)

plt.figure(figsize=(10, 8))
sns.barplot(
    x=top_20_bigrams.values,
    y=top_20_bigrams.index,
    hue=top_20_bigrams.index,
    palette='plasma',
    legend=False
)
plt.title('Top 20 Frasa (Bigrams) yang Paling Sering Muncul')
plt.xlabel('Frekuensi')
plt.ylabel('Bigram')
plt.show()

In [ ]:
print("Memulai Pelabelan Sentimen (Lexicon-Based Manual)")

# 1. Buat Kamus Sentimen (Lexicon) Manual
from arc.lexicon import lexicon_negatif, lexicon_positif

print(f"Kamus sentimen manual dibuat ({len(lexicon_positif)} Pos, {len(lexicon_negatif)} Neg).")

In [ ]:
# 2. Buat Fungsi Pelabelan
def get_sentiment_label_manual(text):
    tokens = text.split()
    score = 0
    negation_words = {'tidak', 'bukan', 'jangan'}
    negate = False

    for word in tokens:
        if word in lexicon_positif:
            score += -1 if negate else 1
            negate = False # Reset negasi
        elif word in lexicon_negatif:
            score += 1 if negate else -1
            negate = False # Reset negasi
        elif word in negation_words:
            negate = True # Aktifkan negasi untuk kata berikutnya
        else:
            negate = False # Reset jika kata netral

    # Tentukan label
    if score > 0:
        return 'Positif'
    elif score < 0:
        return 'Negatif'
    else:
        return 'Netral'

# 3. Terapkan pelabelan ke DataFrame
print("Menerapkan pelabelan sentimen ke semua data...")
# PENTING: Terapkan pada 'final_cleaned_text'
df_processed['sentiment_label'] = df_processed['final_cleaned_text'].apply(get_sentiment_label_manual)
print("--- Pelabelan Sentimen Manual Selesai ---")

In [ ]:
print("Distribusi Label Sentimen (Target y) Baru Anda:")
print(df_processed['sentiment_label'].value_counts())

In [ ]:
# Perbandingan antara rating asli dan label baru
print("Contoh Perbandingan Rating Asli vs Label Baru:")
pd.set_option('display.max_colwidth', 50) # Agar teks tidak terlalu panjang
print(df_processed[['review_text', 'rating', 'final_cleaned_text', 'sentiment_label']].head())

In [ ]:
# Inisialisasi TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1, 2))

X_text = df_processed['final_cleaned_text']     # <-- Ini adalah X
y_labels = df_processed['sentiment_label']      # <-- Ini adalah y (dari lexicon manual)

print("Menerapkan TF-IDF pada kolom 'final_cleaned_text'...")
X_tfidf = tfidf_vectorizer.fit_transform(X_text)

print(f"Bentuk Matriks TF-IDF (X_tfidf): {X_tfidf.shape}")
print(f"Bentuk Target (y_labels): {y_labels.shape}")

In [ ]:
# 1. Split Data (Train & Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_labels,
    test_size=0.2,       # 20% data untuk testing
    random_state=42,     # Agar hasil konsisten
    stratify=y_labels    # Menjaga proporsi Pos/Neg/Net
)
print(f"Data dibagi menjadi {X_train.shape[0]} train dan {X_test.shape[0]} test.")

In [ ]:
# Data untuk pie chart splitting
sizes = [X_train.shape[0], X_test.shape[0]]
labels = [f'Training Data\n{sizes[0]} samples\n({sizes[0]/(sizes[0]+sizes[1])*100:.1f}%)', 
          f'Test Data\n{sizes[1]} samples\n({sizes[1]/(sizes[0]+sizes[1])*100:.1f}%)']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Pie chart untuk train-test split
ax1.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, 
        colors=['#66b3ff','#ff9999'], textprops={'fontsize': 12})
ax1.set_title('Train-Test Data Split (80%-20%)', fontsize=14, fontweight='bold')

# Bar chart untuk distribusi kelas di train dan test
train_counts = y_train.value_counts()
test_counts = y_test.value_counts()

categories = ['Negatif', 'Netral', 'Positif']
x = np.arange(len(categories))
width = 0.35

ax2.bar(x - width/2, [train_counts.get(cat, 0) for cat in categories], 
        width, label='Training', color='#66b3ff', alpha=0.8)
ax2.bar(x + width/2, [test_counts.get(cat, 0) for cat in categories], 
        width, label='Test', color='#ff9999', alpha=0.8)

ax2.set_xlabel('Kelas Sentimen', fontsize=12)
ax2.set_ylabel('Jumlah Sample', fontsize=12)
ax2.set_title('Distribusi Kelas pada Training dan Test Data', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(categories)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Tambah nilai di atas bar
for i, v in enumerate([train_counts.get(cat, 0) for cat in categories]):
    ax2.text(i - width/2, v + 5, str(v), ha='center', va='bottom', fontweight='bold')
for i, v in enumerate([test_counts.get(cat, 0) for cat in categories]):
    ax2.text(i + width/2, v + 5, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
models_results = []

def plot_model_results(y_test, y_pred, model_name, model_index):
    """Fungsi untuk plotting hasil model"""
    
    # Hitung metrics
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Buat figure dengan subplots
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 16))
    fig.suptitle(f'ANALISIS MODEL {model_index}: {model_name.upper()}', 
                 fontsize=16, fontweight='bold', y=0.95)
    
    # ==================== PLOT 1: CONFUSION MATRIX ====================
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Negatif', 'Netral', 'Positif'],
                yticklabels=['Negatif', 'Netral', 'Positif'], ax=ax1)
    ax1.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax1.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax1.set_title(f'Confusion Matrix\nAkurasi: {accuracy:.3f}', 
                  fontsize=14, fontweight='bold')
    
    # ==================== PLOT 2: METRIK PER KELAS ====================
    metrics_data = []
    classes = ['Negatif', 'Netral', 'Positif']
    
    for cls in classes:
        if cls in report:
            metrics_data.append({
                'Kelas': cls,
                'Precision': report[cls]['precision'],
                'Recall': report[cls]['recall'],
                'F1-Score': report[cls]['f1-score'],
                'Support': report[cls]['support']
            })
    
    metrics_df = pd.DataFrame(metrics_data)
    
    x = np.arange(len(classes))
    width = 0.25
    
    ax2.bar(x - width, metrics_df['Precision'], width, label='Precision', 
            color='#2E86AB', alpha=0.8)
    ax2.bar(x, metrics_df['Recall'], width, label='Recall', 
            color='#A23B72', alpha=0.8)
    ax2.bar(x + width, metrics_df['F1-Score'], width, label='F1-Score', 
            color='#F18F01', alpha=0.8)
    
    ax2.set_xlabel('Kelas', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Nilai', fontsize=12, fontweight='bold')
    ax2.set_title('Perbandingan Metrik per Kelas', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(classes)
    ax2.legend()
    ax2.set_ylim(0, 1)
    ax2.grid(axis='y', alpha=0.3)
    
    # Tambah nilai di atas bar
    for i, (p, r, f) in enumerate(zip(metrics_df['Precision'], metrics_df['Recall'], metrics_df['F1-Score'])):
        ax2.text(i - width, p + 0.02, f'{p:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax2.text(i, r + 0.02, f'{r:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax2.text(i + width, f + 0.02, f'{f:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    # ==================== PLOT 3: HEATMAP METRIK ====================
    heatmap_data = metrics_df[['Precision', 'Recall', 'F1-Score']].T
    sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', 
                xticklabels=classes, yticklabels=['Precision', 'Recall', 'F1-Score'],
                ax=ax3, cbar_kws={'label': 'Nilai'})
    ax3.set_title('Heatmap Metrik Evaluasi', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Kelas', fontsize=12, fontweight='bold')
    
    # ==================== PLOT 4: SUPPORT PER KELAS ====================
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    wedges, texts, autotexts = ax4.pie(metrics_df['Support'], 
                                      labels=classes, 
                                      autopct='%1.1f%%',
                                      colors=colors,
                                      startangle=90)
    
    # Style teks dalam pie chart
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(11)
    
    for text in texts:
        text.set_fontsize(12)
        text.set_fontweight('bold')
    
    ax4.set_title('Distribusi Support per Kelas\n(Test Data)', fontsize=14, fontweight='bold')
    
    # Tambah ringkasan metrics di sisi kanan
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    
    summary_text = (
        f"AKURASI: {accuracy:.3f}\n\n"
        f"MACRO AVG:\n"
        f"Precision: {macro_avg['precision']:.3f}\n"
        f"Recall: {macro_avg['recall']:.3f}\n"
        f"F1-Score: {macro_avg['f1-score']:.3f}\n\n"
        f"WEIGHTED AVG:\n"
        f"Precision: {weighted_avg['precision']:.3f}\n"
        f"Recall: {weighted_avg['recall']:.3f}\n"
        f"F1-Score: {weighted_avg['f1-score']:.3f}"
    )
    
    # Tambah text box untuk summary
    fig.text(0.85, 0.25, summary_text, fontsize=12, 
             bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.8),
             verticalalignment='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return report, accuracy

# Parameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# 1. Parameter Tuning untuk Logistic Regression
print("Tuning Logistic Regression...")
param_lr = {
    'C': [0.1, 1, 5, 10, 20, 30, 50, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga']
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_lr.fit(X_train, y_train)

print("Best parameters for Logistic Regression:", grid_lr.best_params_)
best_lr = grid_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test)

print("--- Hasil Setelah Tuning (Logistic Regression) ---")
print(classification_report(y_test, y_pred_lr))

In [ ]:
report_lr, acc_lr = plot_model_results(y_test, y_pred_lr, "Logistic Regression", "I")
models_results.append({
    'Model': 'Logistic Regression',
    'Accuracy': acc_lr,
    'Precision_Macro': report_lr['macro avg']['precision'],
    'Recall_Macro': report_lr['macro avg']['recall'],
    'F1_Macro': report_lr['macro avg']['f1-score'],
    'Precision_Weighted': report_lr['weighted avg']['precision'],
    'Recall_Weighted': report_lr['weighted avg']['recall'],
    'F1_Weighted': report_lr['weighted avg']['f1-score']
})

In [ ]:
# 2. Parameter Tuning untuk Multinomial Naive Bayes
print("\nTuning Multinomial Naive Bayes...")
param_nb = {
    'alpha': [0.1, 0.5, 1.0, 2.0],
    'fit_prior': [True, False]
}

grid_nb = GridSearchCV(
    MultinomialNB(),
    param_nb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_nb.fit(X_train, y_train)

print("Best parameters for Naive Bayes:", grid_nb.best_params_)
best_nb = grid_nb.best_estimator_
y_pred_nb = best_nb.predict(X_test)

print("--- Hasil Setelah Tuning (Naive Bayes) ---")
print(classification_report(y_test, y_pred_nb))

In [ ]:
report_nb, acc_nb = plot_model_results(y_test, y_pred_nb, "Multinomial Naive Bayes", "II")
models_results.append({
    'Model': 'Naive Bayes',
    'Accuracy': acc_nb,
    'Precision_Macro': report_nb['macro avg']['precision'],
    'Recall_Macro': report_nb['macro avg']['recall'],
    'F1_Macro': report_nb['macro avg']['f1-score'],
    'Precision_Weighted': report_nb['weighted avg']['precision'],
    'Recall_Weighted': report_nb['weighted avg']['recall'],
    'F1_Weighted': report_nb['weighted avg']['f1-score']
})

In [ ]:
# 3. Parameter Tuning untuk SVM
print("\nTuning Support Vector Machine...")
param_svm = [
    {
        'kernel': ['linear'],
        'C': [0.1, 1, 10, 100]
    },
    {
        'kernel': ['rbf'],
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.1, 0.01]
    },
    {
        'kernel': ['poly'],
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.1, 0.01],
        'degree': [2, 3, 4]
    },
    {
        'kernel': ['sigmoid'],
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.1, 0.01],
        'coef0': [0, 0.5, 1]
    }
]


grid_svm = GridSearchCV(
    SVC(random_state=42),
    param_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_svm.fit(X_train, y_train)

print("Best parameters for SVM:", grid_svm.best_params_)
best_svm = grid_svm.best_estimator_
y_pred_svm = best_svm.predict(X_test)

print("--- Hasil Setelah Tuning (SVM) ---")
print(classification_report(y_test, y_pred_svm))

In [ ]:
report_svm, acc_svm = plot_model_results(y_test, y_pred_svm, "Support Vector Machine", "III")
models_results.append({
    'Model': 'SVM',
    'Accuracy': acc_svm,
    'Precision_Macro': report_svm['macro avg']['precision'],
    'Recall_Macro': report_svm['macro avg']['recall'],
    'F1_Macro': report_svm['macro avg']['f1-score'],
    'Precision_Weighted': report_svm['weighted avg']['precision'],
    'Recall_Weighted': report_svm['weighted avg']['recall'],
    'F1_Weighted': report_svm['weighted avg']['f1-score']
})

In [ ]:
# Buat DataFrame untuk perbandingan
comparison_df = pd.DataFrame(models_results)

# Plot perbandingan model
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 15))
fig.suptitle('PERBANDINGAN KINERJA TIGA MODEL KLASIFIKASI', 
             fontsize=18, fontweight='bold', y=0.95)

# Warna untuk setiap model
colors = ['#2E86AB', '#A23B72', '#F18F01']

# Plot 1: Akurasi
ax1.bar(comparison_df['Model'], comparison_df['Accuracy'], 
        color=colors, alpha=0.8, edgecolor='black')
ax1.set_ylabel('Akurasi', fontsize=12, fontweight='bold')
ax1.set_title('PERBANDINGAN AKURASI MODEL', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 1)
ax1.grid(axis='y', alpha=0.3)

# Tambah nilai di atas bar
for i, v in enumerate(comparison_df['Accuracy']):
    ax1.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', 
             fontweight='bold', fontsize=11)

# Plot 2: Metrik Macro Average
x = np.arange(len(comparison_df['Model']))
width = 0.25

ax2.bar(x - width, comparison_df['Precision_Macro'], width, 
        label='Precision', color='#2E86AB', alpha=0.8)
ax2.bar(x, comparison_df['Recall_Macro'], width, 
        label='Recall', color='#A23B72', alpha=0.8)
ax2.bar(x + width, comparison_df['F1_Macro'], width, 
        label='F1-Score', color='#F18F01', alpha=0.8)

ax2.set_xlabel('Model', fontsize=12, fontweight='bold')
ax2.set_ylabel('Nilai', fontsize=12, fontweight='bold')
ax2.set_title('METRIK MACRO AVERAGE', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(comparison_df['Model'])
ax2.legend()
ax2.set_ylim(0, 1)
ax2.grid(axis='y', alpha=0.3)

# Plot 3: Metrik Weighted Average
ax3.bar(x - width, comparison_df['Precision_Weighted'], width, 
        label='Precision', color='#2E86AB', alpha=0.8)
ax3.bar(x, comparison_df['Recall_Weighted'], width, 
        label='Recall', color='#A23B72', alpha=0.8)
ax3.bar(x + width, comparison_df['F1_Weighted'], width, 
        label='F1-Score', color='#F18F01', alpha=0.8)

ax3.set_xlabel('Model', fontsize=12, fontweight='bold')
ax3.set_ylabel('Nilai', fontsize=12, fontweight='bold')
ax3.set_title('METRIK WEIGHTED AVERAGE', fontsize=14, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(comparison_df['Model'])
ax3.legend()
ax3.set_ylim(0, 1)
ax3.grid(axis='y', alpha=0.3)

# Plot 4: Heatmap Perbandingan
heatmap_data = comparison_df.set_index('Model')[['Accuracy', 'Precision_Macro', 
                                               'Recall_Macro', 'F1_Macro']]
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlGnBu', 
            ax=ax4, cbar_kws={'label': 'Nilai'})
ax4.set_title('HEATMAP PERBANDINGAN METRIK UTAMA', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Buat tabel perbandingan yang rapi
comparison_table = comparison_df.round(4)
print("\n📋 TABEL PERBANDINGAN MODEL:")
print("-" * 80)
print(comparison_table.to_string(index=False))
print("-" * 80)

# Tentukan model terbaik berdasarkan akurasi
best_model_idx = comparison_df['Accuracy'].idxmax()
best_model = comparison_df.loc[best_model_idx, 'Model']
best_accuracy = comparison_df.loc[best_model_idx, 'Accuracy']

print(f"\n🎯 MODEL TERBAIK: {best_model}")
print(f"📈 AKURASI TERTINGGI: {best_accuracy:.4f}")

In [ ]:
# Perbandingan Model Setelah Tuning
comparison_df_after = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'SVM'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_svm)
    ],
    'Precision_Macro': [
        precision_score(y_test, y_pred_lr, average='macro'),
        precision_score(y_test, y_pred_nb, average='macro'),
        precision_score(y_test, y_pred_svm, average='macro')
    ],
    'Recall_Macro': [
        recall_score(y_test, y_pred_lr, average='macro'),
        recall_score(y_test, y_pred_nb, average='macro'),
        recall_score(y_test, y_pred_svm, average='macro')
    ],
    'F1_Macro': [
        f1_score(y_test, y_pred_lr, average='macro'),
        f1_score(y_test, y_pred_nb, average='macro'),
        f1_score(y_test, y_pred_svm, average='macro')
    ]
})

In [ ]:
# Tampilkan hasil perbandingan
print("\n📋 TABEL PERBANDINGAN SETELAH TUNING:")
print("-" * 80)
print(comparison_df_after.round(4))
print("-" * 80)

# Model terbaik setelah tuning
best_model_after_idx = comparison_df_after['Accuracy'].idxmax()
best_model_after = comparison_df_after.loc[best_model_after_idx, 'Model']
best_accuracy_after = comparison_df_after.loc[best_model_after_idx, 'Accuracy']

print(f"\n🎯 MODEL TERBAIK SETELAH TUNING: {best_model_after}")
print(f"📈 AKURASI TERTINGGI: {best_accuracy_after:.4f}")

In [ ]:
from arc.constants import ASPECT_LIST

from arc.ABSA import lexicon_negatif_absa, lexicon_positif_absa

print(f"Daftar aspek: {ASPECT_LIST}")
print(f"Lexicon positif ABSA dibuat ({len(lexicon_positif_absa)} kata).")
print(f"Lexicon negatif ABSA dibuat ({len(lexicon_negatif_absa)} kata).")
print("--- Persiapan ABSA Selesai ---")


In [ ]:
def get_aspect_sentiment(text, ASPECT_LIST, lexicon_positif, lexicon_negatif, window_size=3):
    tokens = text.split()
    aspect_sentiments = {}

    for aspect in ASPECT_LIST:

        aspect_found = False
        aspect_score = 0
        negate = False

        # Coba stem aspek juga untuk pencarian yang lebih akurat
        stemmed_aspect = stemmer.stem(aspect)

        for i, token in enumerate(tokens):
            if stemmed_aspect == token or aspect == token: # Cek aspek asli atau yang distemmed
                aspect_found = True

                # Ambil konteks sekitar aspek
                start_idx = max(0, i - window_size)
                end_idx = min(len(tokens), i + window_size + 1)
                context_tokens = tokens[start_idx:i] + tokens[i+1:end_idx]

                # Hitung sentimen dalam konteks
                for word in context_tokens:
                    if word in lexicon_positif:
                        aspect_score += -1 if negate else 1
                        negate = False
                    elif word in lexicon_negatif:
                        aspect_score += 1 if negate else -1
                        negate = False
                    elif word in {'tidak', 'bukan', 'jangan'}: # Kata negasi
                        negate = True
                    else:
                        negate = False
                break # Asumsi kita hanya perlu sentimen pertama kali aspek muncul

        if aspect_found:
            if aspect_score > 0:
                aspect_sentiments[aspect] = 'Positif'
            elif aspect_score < 0:
                aspect_sentiments[aspect] = 'Negatif'
            else:
                aspect_sentiments[aspect] = 'Netral'
        else:
            aspect_sentiments[aspect] = 'Tidak Disebutkan'

    return aspect_sentiments

In [ ]:
from IPython.display import display

print("Menerapkan analisis sentimen berbasis aspek...")
df_processed['aspect_sentiments'] = df_processed['final_cleaned_text'].apply(
    lambda x: get_aspect_sentiment(x, ASPECT_LIST, lexicon_positif_absa, lexicon_negatif_absa)
)

print("--- Analisis Sentimen Berbasis Aspek Selesai ---")
print("Contoh hasil analisis sentimen berbasis aspek:")
display(df_processed[['review_text', 'final_cleaned_text', 'aspect_sentiments']].head())

In [ ]:
# Analisis distribusi sentimen untuk setiap aspek
print("Distribusi Sentimen untuk Setiap Aspek:")
aspect_sentiment_counts = {aspect: Counter() for aspect in ASPECT_LIST}

for index, row in df_processed.iterrows():
    for aspect, sentiment in row['aspect_sentiments'].items():
        if aspect in aspect_sentiment_counts:
            aspect_sentiment_counts[aspect][sentiment] += 1

for aspect, counts in aspect_sentiment_counts.items():
    print(f"\nAspek: {aspect}")
    for sentiment, count in counts.most_common():
        print(f"  {sentiment}: {count}")


In [ ]:
# ===========================================================
# 1. SIAPKAN DATA
# ===========================================================
import pandas as pd
from collections import Counter

# data hasil looping Anda (bisa langsung di-overwrite)
aspect_sentiment_counts = {
    'keamanan':  Counter({'Tidak Disebutkan': 2771, 'Netral': 19, 'Positif': 19, 'Negatif': 7}),
    'kebersihan':Counter({'Tidak Disebutkan': 2467, 'Netral': 146, 'Positif': 126, 'Negatif': 77}),
    'keindahan': Counter({'Tidak Disebutkan': 2508, 'Netral': 170, 'Positif': 107, 'Negatif': 31}),
    'fasilitas': Counter({'Tidak Disebutkan': 2713, 'Netral': 37, 'Negatif': 34, 'Positif': 32}),
    'aksesibilitas': Counter({'Tidak Disebutkan': 2816})
}

# ubah ke DataFrame biar gampang dipakai seaborn
df_list = []
for aspect, cnt in aspect_sentiment_counts.items():
    for sent, val in cnt.items():
        df_list.append({'Aspek': aspect, 'Sentimen': sent, 'Jumlah': val})
df = pd.DataFrame(df_list)

# ===========================================================
# 2. VISUALISASI 1 – STACKED-BAR (JUMLAH)
#    cocok untuk semua kalangan
# ===========================================================
import seaborn as sns
import matplotlib.pyplot as plt

# warna konsisten utk tiap sentimen
palette = {'Positif':'#2E8B57',    # hijau tua
           'Netral' :'#7F7F7F',    # abu
           'Negatif':'#B22222',    # merah bata
           'Tidak Disebutkan':'#E0E0E0'} # abu muda

plt.figure(figsize=(10,5))
sns.barplot(data=df,
            x='Aspek',
            y='Jumlah',
            hue='Sentimen',
            palette=palette,
            dodge=False,
            estimator=sum, ci=None)
plt.title('Distribusi Sentimen untuk Tiap Aspek (Jumlah)', fontsize=14)
plt.ylabel('Banyak ulasan')
plt.xlabel('')
plt.legend(title='Sentimen', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()

# ===========================================================
# 3. VISUALISASI 2 – 100 % STACKED-BAR (PROPORSI)
#    sangat mudah dibaca oleh non-teknisi
# ===========================================================
prop_df = (df.assign(Jumlah=lambda d: d.groupby(['Aspek', 'Sentimen'])['Jumlah'].transform('sum'))
             .pivot_table(index='Aspek', columns='Sentimen', values='Jumlah', fill_value=0))
prop_df = prop_df.div(prop_df.sum(axis=1), axis=0)   # jadikan proporsi

ax = prop_df.plot(kind='bar', stacked=True,
                  color=[palette[col] for col in prop_df.columns],
                  figsize=(10,5))
plt.title('Proporsi Sentimen untuk Tiap Aspek (%)', fontsize=14)
plt.ylabel('Proporsi')
plt.xlabel('')
plt.legend(title='Sentimen', bbox_to_anchor=(1.02,1), loc='upper left')
plt.xticks(rotation=0)
for c in ax.containers:
    # tambahkan label % jika > 5 %
    ax.bar_label(c, fmt='%.0f%%', label_type='center',
                 fontsize=8, fontweight='bold',
                 labels=[f'{v*100:.0f}%' if v>0.05 else '' for v in c.datavalues])
plt.tight_layout()
plt.show()

# ===========================================================
# 4. VISUALISASI 3 – HEATMAP (utk data scientist)
#    melihat pola outlier sekilas
# ===========================================================
plt.figure(figsize=(6,3))
sns.heatmap(prop_df, annot=True, fmt='.1%', cmap='RdYlGn_r',
            cbar_kws={'label': 'Proporsi'})
plt.title('Heatmap Proporsi Sentimen per Aspek')
plt.ylabel('')
plt.tight_layout()
plt.show()